# Part 4 — Final 50,000-comment structured extraction

## Frozen production method

- **Method version:** `airbnb_extraction_v1.0`
- **Model:** `gpt-5-mini-2025-08-07`
- **Reasoning:** `medium`
- **Request design:** 10 comments per request
- **Concurrency:** 10 requests at a time
- **Prompt/schema source:** project-root `extraction_config.py`
- **Master input:** `data/samples/reviews_sample_50000_extraction_ready.csv`
- **Partitions:** five fixed files of 10,000 comments in `data/final_parts/`
- **Selected part:** change `PART_NUMBER` to 1, 2, 3, 4, or 5
- **Outputs:** separate checkpoints in `outputs/final_50000/part_01/` through `part_05/`
- **Final merge:** created in `outputs/final_50000/merged/` only after all parts pass audit
- **Checkpointing:** every successful batch is written immediately
- **Resume behavior:** completed comment IDs in the selected part are skipped
- **Budget:** one shared $80 safety limit across all five parts
- **Attempts:** up to 3 total per request (1 initial attempt + 2 retries)

Run and audit one part at a time. After this method is frozen, do not change the prompt, taxonomy, schema, model, reasoning level, batch size, concurrency, partition files, or part boundaries during the 50,000-comment run.


## Imports and frozen production settings


In [ ]:
from concurrent.futures import ThreadPoolExecutor, as_completed
from datetime import datetime, timezone
from getpass import getpass
from pathlib import Path
from pydantic import BaseModel, ConfigDict, Field
import hashlib
import importlib
import json
import os
import random
import sys
import time

import openai
import pandas as pd
from openai import OpenAI


METHOD_VERSION = "airbnb_extraction_v1.0"
PART_NUMBER = 5  # Change only this value: 1, 2, 3, 4, or 5.
TOTAL_PARTS = 5
ROWS_PER_PART = 10_000
EXPECTED_TOTAL_ROWS = TOTAL_PARTS * ROWS_PER_PART
MODEL = "gpt-5-mini-2025-08-07"
REASONING_EFFORT = "medium"
COMMENTS_PER_REQUEST = 10
MAX_WORKERS = 10
MAX_ATTEMPTS = 3
RETRY_BASE_DELAY_SECONDS = 2.0
RETRY_MAX_BACKOFF_SECONDS = 60.0
RETRY_JITTER_SECONDS = 1.0
REQUEST_TIMEOUT_SECONDS = 600.0
REQUESTS_PER_WAVE = 10  # 100 comments per budget-checked wave
MAX_TOTAL_COST_USD = 80.00
MIN_BUDGET_RESERVE_PER_REQUEST_USD = 0.025
OBSERVED_COST_RESERVE_MULTIPLIER = 1.25

if not 1 <= PART_NUMBER <= TOTAL_PARTS:
    raise ValueError(f"PART_NUMBER must be from 1 to {TOTAL_PARTS}.")

# GPT-5 mini standard token prices used for run tracking.
INPUT_PRICE_PER_M = 0.25
CACHED_INPUT_PRICE_PER_M = 0.025
OUTPUT_PRICE_PER_M = 2.00

# Frozen local method files, with line endings normalized before hashing.
EXPECTED_CONFIG_SHA256 = (
    "364430c438c0562829adf0a8f49577c49c2896791742655b25f318c4a8d7dbcb"
)
EXPECTED_WORKFLOW_SHA256 = (
    "668342a8db60240f906456b4ffced9aaf6b26fd908cc0a2f6bc988c43901eecb"
)

BASE_DIR = Path.cwd().resolve()
MASTER_INPUT_FILE = (
    BASE_DIR
    / "data"
    / "samples"
    / "reviews_sample_50000_extraction_ready.csv"
)
PARTS_DIR = BASE_DIR / "data" / "final_parts"
INPUT_FILE = PARTS_DIR / f"reviews_part_{PART_NUMBER:02d}.csv"
CONFIG_FILE = BASE_DIR / "extraction_config.py"
WORKFLOW_FILE = BASE_DIR / "src" / "final_extraction_workflow.py"

OUTPUT_ROOT = BASE_DIR / "outputs" / "final_50000"
OUTPUT_DIR = OUTPUT_ROOT / f"part_{PART_NUMBER:02d}"
MERGED_OUTPUT_DIR = OUTPUT_ROOT / "merged"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

PART_PREFIX = f"part_{PART_NUMBER:02d}"
RESULTS_FILE = OUTPUT_DIR / f"{PART_PREFIX}_extractions.jsonl"
ERRORS_FILE = OUTPUT_DIR / f"{PART_PREFIX}_errors.jsonl"
USAGE_FILE = OUTPUT_DIR / f"{PART_PREFIX}_usage.jsonl"
MANIFEST_FILE = OUTPUT_DIR / f"{PART_PREFIX}_run_manifest.json"
FINDINGS_FILE = OUTPUT_DIR / f"{PART_PREFIX}_findings.csv"
REVIEW_SUMMARY_FILE = OUTPUT_DIR / f"{PART_PREFIX}_review_summary.csv"
AUDIT_FILE = OUTPUT_DIR / f"{PART_PREFIX}_audit.json"
EXPECTED_CONFIRMATION = f"RUN PART {PART_NUMBER}"


if not CONFIG_FILE.exists():
    raise FileNotFoundError(
        "Cannot find extraction_config.py in the project root:\n"
        f"{CONFIG_FILE}"
    )
if not WORKFLOW_FILE.exists():
    raise FileNotFoundError(
        "Cannot find src/final_extraction_workflow.py:\n"
        f"{WORKFLOW_FILE}"
    )

config_text = CONFIG_FILE.read_text(encoding="utf-8")
config_text = config_text.replace("\r\n", "\n").replace("\r", "\n")
actual_config_sha256 = hashlib.sha256(
    config_text.encode("utf-8")
).hexdigest()

if actual_config_sha256 != EXPECTED_CONFIG_SHA256:
    raise RuntimeError(
        "The project-root extraction_config.py is not the exact frozen version.\n"
        f"Expected SHA-256: {EXPECTED_CONFIG_SHA256}\n"
        f"Actual SHA-256:   {actual_config_sha256}"
    )

workflow_text = WORKFLOW_FILE.read_text(encoding="utf-8")
workflow_text = workflow_text.replace("\r\n", "\n").replace("\r", "\n")
actual_workflow_sha256 = hashlib.sha256(
    workflow_text.encode("utf-8")
).hexdigest()
if actual_workflow_sha256 != EXPECTED_WORKFLOW_SHA256:
    raise RuntimeError(
        "The production workflow helper is not the frozen version.\n"
        f"Expected SHA-256: {EXPECTED_WORKFLOW_SHA256}\n"
        f"Actual SHA-256:   {actual_workflow_sha256}"
    )

# Force Python to import the configuration from this project folder.
if str(BASE_DIR) not in sys.path:
    sys.path.insert(0, str(BASE_DIR))

importlib.invalidate_caches()
sys.modules.pop("extraction_config", None)
extraction_config = importlib.import_module("extraction_config")

loaded_config_path = Path(extraction_config.__file__).resolve()
if loaded_config_path != CONFIG_FILE.resolve():
    raise RuntimeError(
        "Python loaded a different extraction_config.py file.\n"
        f"Expected: {CONFIG_FILE.resolve()}\n"
        f"Loaded:   {loaded_config_path}"
    )

CommentExtraction = extraction_config.CommentExtraction
EXTRACTION_PROMPT_V1 = extraction_config.EXTRACTION_PROMPT_V1
from src.final_extraction_workflow import (
    PARTITION_VERSION,
    estimate_wave_cost_reserve,
    load_global_usage,
    merge_final_parts,
    prepare_final_partitions,
)

print("Project folder:", BASE_DIR)
print("Frozen method:", METHOD_VERSION)
print("Selected part:", f"{PART_NUMBER}/{TOTAL_PARTS}")
print("Model:", MODEL)
print("Reasoning:", REASONING_EFFORT)
print("Comments per request:", COMMENTS_PER_REQUEST)
print("Parallel requests:", MAX_WORKERS)
print("Configuration:", loaded_config_path)
print("Configuration SHA-256:", actual_config_sha256)
print("Workflow SHA-256:", actual_workflow_sha256)
print("Master input file:", MASTER_INPUT_FILE)
print("Selected part file:", INPUT_FILE)
print("Output folder:", OUTPUT_DIR)


Project folder: C:\Users\yang9\Desktop\Capstone 406\Airbnb\python2
Frozen method: airbnb_extraction_v1.0
Selected part: 4/5
Model: gpt-5-mini-2025-08-07
Reasoning: medium
Comments per request: 10
Parallel requests: 10
Configuration: C:\Users\yang9\Desktop\Capstone 406\Airbnb\python2\extraction_config.py
Configuration SHA-256: 364430c438c0562829adf0a8f49577c49c2896791742655b25f318c4a8d7dbcb
Workflow SHA-256: 668342a8db60240f906456b4ffced9aaf6b26fd908cc0a2f6bc988c43901eecb
Master input file: C:\Users\yang9\Desktop\Capstone 406\Airbnb\python2\data\samples\reviews_sample_50000_extraction_ready.csv
Selected part file: C:\Users\yang9\Desktop\Capstone 406\Airbnb\python2\data\final_parts\reviews_part_04.csv
Output folder: C:\Users\yang9\Desktop\Capstone 406\Airbnb\python2\outputs\final_50000\part_04


## Validate the master input and select one frozen 10,000-comment part

In [2]:
if not MASTER_INPUT_FILE.exists():
    raise FileNotFoundError(
        f"Cannot find {MASTER_INPUT_FILE.name}. Run 02_sampling.ipynb first."
    )

if not CONFIG_FILE.exists():
    raise FileNotFoundError(
        f"Cannot find {CONFIG_FILE.name} in {BASE_DIR}."
    )

required_prompt_checks = {
    "Aesthetics and design": "Aesthetics and design" in EXTRACTION_PROMPT_V1,
    "Space decision": (
        'Phrases such as "great space" or "amazing space" evaluate'
        in EXTRACTION_PROMPT_V1
        and "the physical space of the accommodation and belong to"
        in EXTRACTION_PROMPT_V1
    ),
    "Pure gratitude rule": "Pure gratitude" in EXTRACTION_PROMPT_V1,
    "Negative accuracy mismatch": (
        "Any factual or material mismatch is negative" in EXTRACTION_PROMPT_V1
    ),
}

failed_prompt_checks = [
    name for name, passed in required_prompt_checks.items() if not passed
]
if failed_prompt_checks:
    raise RuntimeError(
        "The loaded extraction_config.py is not the frozen final version. "
        f"Failed checks: {failed_prompt_checks}"
    )

required_columns = {
    "listing_id",
    "id",
    "date",
    "year",
    "listing_comment_count",
    "listing_activity",
    "comments_original",
    "comments_clean",
}

df_master = pd.read_csv(
    MASTER_INPUT_FILE,
    dtype={"id": "string", "listing_id": "string"},
)
missing_columns = sorted(required_columns - set(df_master.columns))
if missing_columns:
    raise ValueError(f"Missing required columns: {missing_columns}")

if len(df_master) != EXPECTED_TOTAL_ROWS:
    raise ValueError(
        f"Expected {EXPECTED_TOTAL_ROWS:,} master rows, "
        f"but found {len(df_master):,}."
    )

if df_master["id"].isna().any():
    raise ValueError("The master input contains missing comment IDs.")

if df_master["id"].duplicated().any():
    duplicate_count = int(df_master["id"].duplicated().sum())
    raise ValueError(
        f"The master input contains {duplicate_count} duplicate comment IDs."
    )

if df_master["comments_clean"].isna().any():
    raise ValueError("The master input contains missing cleaned comments.")

if df_master["comments_clean"].astype(str).str.strip().eq("").any():
    raise ValueError("The master input contains empty cleaned comments.")

# Freeze one global source-row sequence before making equal partitions.
df_master = df_master.reset_index(drop=True)
df_master["source_row"] = df_master.index + 1
partition_manifest = prepare_final_partitions(
    df_master,
    MASTER_INPUT_FILE,
    PARTS_DIR,
    total_parts=TOTAL_PARTS,
    rows_per_part=ROWS_PER_PART,
)

df_final = pd.read_csv(
    INPUT_FILE,
    dtype={"id": "string", "listing_id": "string"},
)
if len(df_final) != ROWS_PER_PART:
    raise ValueError(
        f"Part {PART_NUMBER} must contain {ROWS_PER_PART:,} rows; "
        f"found {len(df_final):,}."
    )
if df_final["id"].isna().any() or df_final["id"].duplicated().any():
    raise ValueError(f"Part {PART_NUMBER} has missing or duplicate IDs.")
if df_final["comments_clean"].isna().any():
    raise ValueError(f"Part {PART_NUMBER} has missing cleaned comments.")
if df_final["comments_clean"].astype(str).str.strip().eq("").any():
    raise ValueError(f"Part {PART_NUMBER} has empty cleaned comments.")

expected_start = (PART_NUMBER - 1) * ROWS_PER_PART
expected_end = PART_NUMBER * ROWS_PER_PART
expected_part_ids = list(
    df_master.iloc[expected_start:expected_end]["id"].astype(str)
)
if list(df_final["id"].astype(str)) != expected_part_ids:
    raise RuntimeError(
        f"Part {PART_NUMBER} IDs or ordering do not match the master input."
    )
expected_source_rows = list(range(expected_start + 1, expected_end + 1))
if list(pd.to_numeric(df_final["source_row"], errors="raise")) != expected_source_rows:
    raise RuntimeError(f"Part {PART_NUMBER} source-row boundaries are invalid.")
df_final["source_row"] = pd.to_numeric(
    df_final["source_row"], errors="raise"
).astype(int)

print("Master rows:", len(df_master))
print("Partition version:", PARTITION_VERSION)
print("Selected part:", f"{PART_NUMBER}/{TOTAL_PARTS}")
print("Part rows:", len(df_final))
print("Part unique comment IDs:", df_final["id"].nunique())
print("Part source rows:", f"{expected_start + 1:,}-{expected_end:,}")
print("Prompt checks:", required_prompt_checks)
print("Input file:", INPUT_FILE)


Master rows: 50000
Partition version: airbnb_final_50000_five_equal_parts_v1
Selected part: 4/5
Part rows: 10000
Part unique comment IDs: 10000
Part source rows: 30,001-40,000
Prompt checks: {'Aesthetics and design': True, 'Space decision': True, 'Pure gratitude rule': True, 'Negative accuracy mismatch': True}
Input file: C:\Users\yang9\Desktop\Capstone 406\Airbnb\python2\data\final_parts\reviews_part_04.csv


## Batch schema, cost helpers, hashing, and API client


In [3]:
#A hashing mechanism generates a unique, fixed‑length fingerprint for data.
class CommentBatchExtraction(BaseModel):
    model_config = ConfigDict(extra="forbid")

    extractions: list[CommentExtraction] = Field(
        min_length=1,
        max_length=COMMENTS_PER_REQUEST,
        description="Exactly one independent extraction for every input comment.",
    )


BATCH_INSTRUCTION = """

MULTI-COMMENT REQUEST RULES

The user message contains a JSON object with a "comments" list.
Process each comment independently using all rules above.

Return exactly one CommentExtraction for every input comment.
Copy every comment_id exactly.
Do not omit, duplicate, merge, or combine comments.
Do not use information from one comment to interpret another.
Return the extractions in the same order as the input comments.
"""

SYSTEM_PROMPT_BATCH = EXTRACTION_PROMPT_V1 + BATCH_INSTRUCTION


def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as file:
        for chunk in iter(lambda: file.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def sha256_text(text: str) -> str:
    return hashlib.sha256(text.encode("utf-8")).hexdigest()


def nested_value(obj, *names, default=0):
    current = obj
    for name in names:
        if current is None:
            return default
        current = getattr(current, name, None)
    return default if current is None else current


def calculate_cost(usage: dict) -> float:
    uncached_input = max(
        usage["input_tokens"] - usage["cached_input_tokens"],
        0,
    )
    return (
        uncached_input * INPUT_PRICE_PER_M / 1_000_000
        + usage["cached_input_tokens"] * CACHED_INPUT_PRICE_PER_M / 1_000_000
        + usage["output_tokens"] * OUTPUT_PRICE_PER_M / 1_000_000
    )


def read_jsonl(path: Path) -> list[dict]:
    records = []
    if not path.exists():
        return records
    with path.open("r", encoding="utf-8") as file:
        for line_number, line in enumerate(file, start=1):
            if not line.strip():
                continue
            try:
                records.append(json.loads(line))
            except json.JSONDecodeError as error:
                raise ValueError(
                    f"Invalid JSON on line {line_number} of {path.name}: {error}"
                ) from error
    return records


def make_batches(rows: list[dict], batch_size: int) -> list[dict]:
    batches = []
    for start in range(0, len(rows), batch_size):
        group = rows[start : start + batch_size]
        first_row = int(group[0]["source_row"])
        last_row = int(group[-1]["source_row"])
        batches.append(
            {
                "batch_id": f"rows_{first_row:06d}_{last_row:06d}",
                "rows": group,
            }
        )
    return batches


ACTION_REQUIRED_ERROR_CODES = {
    "billing_hard_limit_reached",
    "billing_not_active",
    "insufficient_quota",
}

NON_RETRYABLE_API_ERRORS = (
    openai.AuthenticationError,
    openai.BadRequestError,
    openai.NotFoundError,
    openai.PermissionDeniedError,
    openai.UnprocessableEntityError,
)

RETRYABLE_API_ERRORS = (
    openai.APIConnectionError,
    openai.APITimeoutError,
    openai.InternalServerError,
    openai.RateLimitError,
)


def api_error_code(error: Exception) -> str | None:
    candidates = [
        getattr(error, "code", None),
        getattr(error, "type", None),
    ]
    body = getattr(error, "body", None)
    if isinstance(body, dict):
        details = body.get("error", body)
        if isinstance(details, dict):
            candidates.extend(
                [details.get("code"), details.get("type")]
            )

    for candidate in candidates:
        if candidate:
            return str(candidate).lower()
    return None


def error_metadata(error: Exception) -> dict:
    return {
        "error_type": type(error).__name__,
        "message": str(error),
        "status_code": getattr(error, "status_code", None),
        "request_id": getattr(error, "request_id", None),
        "api_error_code": api_error_code(error),
    }


def is_retryable_error(error: Exception) -> bool:
    # A structurally valid request can still return a mismatched batch;
    # retrying can recover from that model-response validation failure.
    if isinstance(error, ValueError):
        return True

    if isinstance(error, NON_RETRYABLE_API_ERRORS):
        return False

    if api_error_code(error) in ACTION_REQUIRED_ERROR_CODES:
        return False

    if isinstance(error, RETRYABLE_API_ERRORS):
        return True

    if isinstance(error, openai.APIStatusError):
        status_code = getattr(error, "status_code", None)
        return (
            status_code in {408, 409, 429}
            or (status_code is not None and status_code >= 500)
        )

    return False


def retry_after_seconds(error: Exception) -> float | None:
    response = getattr(error, "response", None)
    headers = getattr(response, "headers", None)
    if headers is None:
        return None

    value = headers.get("retry-after")
    if value is None:
        return None

    try:
        seconds = float(value)
    except (TypeError, ValueError):
        return None

    return seconds if seconds >= 0 else None


def calculate_retry_delay(error: Exception, attempt: int) -> float:
    exponential_delay = min(
        RETRY_MAX_BACKOFF_SECONDS,
        RETRY_BASE_DELAY_SECONDS * (2 ** (attempt - 1)),
    )
    server_delay = retry_after_seconds(error)
    minimum_delay = max(exponential_delay, server_delay or 0.0)
    return minimum_delay + random.uniform(0.0, RETRY_JITTER_SECONDS)


if not os.getenv("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API key: ")

# Disable hidden SDK retries so the three-attempt policy stays exact and logged.
client = OpenAI(max_retries=0, timeout=REQUEST_TIMEOUT_SECONDS)
print("API client created.")


API client created.


## Request function: one request processes up to 10 comments


In [4]:
def extract_one_batch(batch: dict) -> dict:
    rows = batch["rows"]
    batch_id = batch["batch_id"]
    expected_ids = [str(row["id"]) for row in rows]

    payload = {
        "comments": [
            {
                "comment_id": str(row["id"]),
                "comment": str(row["comments_clean"]),
            }
            for row in rows
        ]
    }

    attempt_history = []

    for attempt in range(1, MAX_ATTEMPTS + 1):
        started = time.perf_counter()

        try:
            completion = client.chat.completions.parse(
                model=MODEL,
                reasoning_effort=REASONING_EFFORT,
                messages=[
                    {"role": "system", "content": SYSTEM_PROMPT_BATCH},
                    {
                        "role": "user",
                        "content": json.dumps(payload, ensure_ascii=False),
                    },
                ],
                response_format=CommentBatchExtraction,
            )

            elapsed_seconds = time.perf_counter() - started
            message = completion.choices[0].message

            if message.parsed is None:
                raise ValueError(
                    f"No parsed batch extraction returned. Refusal: {message.refusal}"
                )

            parsed_batch = message.parsed
            returned_ids = [
                str(item.comment_id) for item in parsed_batch.extractions
            ]

            if len(returned_ids) != len(expected_ids):
                raise ValueError(
                    f"Expected {len(expected_ids)} extractions, "
                    f"but received {len(returned_ids)}."
                )

            if len(set(returned_ids)) != len(returned_ids):
                raise ValueError("The response contains duplicate comment IDs.")

            if set(returned_ids) != set(expected_ids):
                missing = sorted(set(expected_ids) - set(returned_ids))
                unexpected = sorted(set(returned_ids) - set(expected_ids))
                raise ValueError(
                    f"Comment ID mismatch. Missing={missing}; unexpected={unexpected}."
                )

            extraction_by_id = {
                str(item.comment_id): item for item in parsed_batch.extractions
            }

            usage_object = completion.usage
            usage = {
                "input_tokens": nested_value(usage_object, "prompt_tokens"),
                "cached_input_tokens": nested_value(
                    usage_object,
                    "prompt_tokens_details",
                    "cached_tokens",
                ),
                "output_tokens": nested_value(usage_object, "completion_tokens"),
                "reasoning_tokens": nested_value(
                    usage_object,
                    "completion_tokens_details",
                    "reasoning_tokens",
                ),
                "total_tokens": nested_value(usage_object, "total_tokens"),
            }
            usage["estimated_cost_usd"] = calculate_cost(usage)

            comment_records = []
            for row in rows:
                comment_id = str(row["id"])
                extraction = extraction_by_id[comment_id]
                comment_records.append(
                    {
                        "source_row": int(row["source_row"]),
                        "part_number": PART_NUMBER,
                        "total_parts": TOTAL_PARTS,
                        "listing_id": str(row["listing_id"]),
                        "comment_id": comment_id,
                        "date": row["date"],
                        "year": int(row["year"]),
                        "listing_comment_count": int(row["listing_comment_count"]),
                        "listing_activity": row["listing_activity"],
                        "comments_original": str(row["comments_original"]),
                        "comments_clean": str(row["comments_clean"]),
                        "method_version": METHOD_VERSION,
                        "run_signature_sha256": run_signature_sha256,
                        "model": MODEL,
                        "reasoning_effort": REASONING_EFFORT,
                        "comments_per_request": len(rows),
                        "batch_id": batch_id,
                        "attempt": attempt,
                        "extraction": extraction.model_dump(),
                    }
                )

            return {
                "status": "success",
                "batch_id": batch_id,
                "comment_records": comment_records,
                "usage_record": {
                    "batch_id": batch_id,
                    "part_number": PART_NUMBER,
                    "total_parts": TOTAL_PARTS,
                    "run_signature_sha256": run_signature_sha256,
                    "source_rows": [int(row["source_row"]) for row in rows],
                    "comment_ids": expected_ids,
                    "comment_count": len(rows),
                    "elapsed_seconds": elapsed_seconds,
                    "attempt": attempt,
                    "retry_history": attempt_history,
                    "model": MODEL,
                    "reasoning_effort": REASONING_EFFORT,
                    "usage": usage,
                },
            }

        except Exception as error:
            retryable = is_retryable_error(error)
            attempt_record = {
                "attempt": attempt,
                "retryable": retryable,
                **error_metadata(error),
            }

            if retryable and attempt < MAX_ATTEMPTS:
                wait_seconds = calculate_retry_delay(error, attempt)
                attempt_record["wait_seconds"] = round(wait_seconds, 3)
                attempt_history.append(attempt_record)
                print(
                    f"Retrying {batch_id}: attempt {attempt}/{MAX_ATTEMPTS} "
                    f"failed with {type(error).__name__}; "
                    f"waiting {wait_seconds:.1f} seconds."
                )
                time.sleep(wait_seconds)
                continue

            attempt_history.append(attempt_record)
            return {
                "status": "error" if retryable else "fatal_error",
                "batch_id": batch_id,
                "part_number": PART_NUMBER,
                "total_parts": TOTAL_PARTS,
                "run_signature_sha256": run_signature_sha256,
                "source_rows": [int(row["source_row"]) for row in rows],
                "comment_ids": expected_ids,
                "error_type": type(error).__name__,
                "message": str(error),
                "status_code": getattr(error, "status_code", None),
                "request_id": getattr(error, "request_id", None),
                "api_error_code": api_error_code(error),
                "retryable": retryable,
                "attempts": attempt,
                "attempt_history": attempt_history,
            }


## Build the selected part's resumable run plan and validate every checkpoint.
### No comment ID is skipped until all validation below has passed.

In [5]:
schema_sha256 = sha256_text(
    json.dumps(
        CommentExtraction.model_json_schema(),
        sort_keys=True,
        separators=(",", ":"),
    )
)

run_signature = {
    "method_version": METHOD_VERSION,
    "partition_version": PARTITION_VERSION,
    "part_number": PART_NUMBER,
    "total_parts": TOTAL_PARTS,
    "rows_per_part": ROWS_PER_PART,
    "model": MODEL,
    "reasoning_effort": REASONING_EFFORT,
    "comments_per_request": COMMENTS_PER_REQUEST,
    "max_workers": MAX_WORKERS,
    "max_attempts": MAX_ATTEMPTS,
    "request_timeout_seconds": REQUEST_TIMEOUT_SECONDS,
    "requests_per_wave": REQUESTS_PER_WAVE,
    "input_rows": len(df_final),
    "config_sha256": actual_config_sha256,
    "workflow_sha256": actual_workflow_sha256,
    "master_input_sha256": partition_manifest["master_sha256"],
    "input_sha256": sha256_file(INPUT_FILE),
    "prompt_sha256": sha256_text(SYSTEM_PROMPT_BATCH),
    "schema_sha256": schema_sha256,
}
run_signature_sha256 = sha256_text(
    json.dumps(run_signature, sort_keys=True, separators=(",", ":"))
)

checkpoint_has_data = any(
    path.exists() and path.stat().st_size > 0
    for path in (RESULTS_FILE, ERRORS_FILE, USAGE_FILE)
)

resume_manifest = None
resume_error_state = {"count": 0, "examples": []}


def add_resume_error(message: str) -> None:
    resume_error_state["count"] += 1
    if len(resume_error_state["examples"]) < 50:
        resume_error_state["examples"].append(message)


if MANIFEST_FILE.exists():
    try:
        resume_manifest = json.loads(
            MANIFEST_FILE.read_text(encoding="utf-8")
        )
    except json.JSONDecodeError as error:
        add_resume_error(
            f"Manifest is not valid JSON: {error}"
        )
    else:
        if not isinstance(resume_manifest, dict):
            add_resume_error("Manifest must contain one JSON object.")
        else:
            for field, expected_value in run_signature.items():
                actual_value = resume_manifest.get(field, "<missing>")
                if actual_value != expected_value:
                    add_resume_error(
                        f"Manifest {field} mismatch: "
                        f"saved={actual_value!r}; current={expected_value!r}"
                    )

            saved_signature = resume_manifest.get(
                "run_signature_sha256"
            )
            if saved_signature != run_signature_sha256:
                add_resume_error(
                    "Manifest run signature does not match the current run. "
                    f"saved={saved_signature!r}; "
                    f"current={run_signature_sha256!r}"
                )
elif checkpoint_has_data:
    add_resume_error(
        "Checkpoint data exists but the run manifest is missing."
    )

saved_results = read_jsonl(RESULTS_FILE)
existing_usage = read_jsonl(USAGE_FILE)
input_by_id = {
    str(row["id"]): row
    for row in df_final.to_dict("records")
}

completed_ids = set()
result_ids_by_batch = {}
result_rows_by_batch = {}

for line_number, record in enumerate(saved_results, start=1):
    if not isinstance(record, dict):
        add_resume_error(
            f"Results line {line_number} is not a JSON object."
        )
        continue

    raw_comment_id = record.get("comment_id")
    if raw_comment_id is None:
        add_resume_error(
            f"Results line {line_number} has no comment_id."
        )
        continue

    comment_id = str(raw_comment_id)
    if comment_id in completed_ids:
        add_resume_error(
            f"Duplicate saved comment_id {comment_id!r} "
            f"on results line {line_number}."
        )
        continue

    expected_row = input_by_id.get(comment_id)
    if expected_row is None:
        add_resume_error(
            f"Unexpected comment_id {comment_id!r} "
            f"on results line {line_number}."
        )
        continue

    if record.get("run_signature_sha256") != run_signature_sha256:
        add_resume_error(
            f"Result {comment_id} has a different or missing run signature."
        )

    expected_metadata = {
        "source_row": int(expected_row["source_row"]),
        "part_number": PART_NUMBER,
        "total_parts": TOTAL_PARTS,
        "listing_id": str(expected_row["listing_id"]),
        "date": str(expected_row["date"]),
        "year": int(expected_row["year"]),
        "listing_comment_count": int(
            expected_row["listing_comment_count"]
        ),
        "listing_activity": str(expected_row["listing_activity"]),
        "comments_original": str(expected_row["comments_original"]),
        "comments_clean": str(expected_row["comments_clean"]),
        "method_version": METHOD_VERSION,
        "model": MODEL,
        "reasoning_effort": REASONING_EFFORT,
    }
    for field, expected_value in expected_metadata.items():
        if record.get(field) != expected_value:
            add_resume_error(
                f"Result {comment_id} {field} mismatch: "
                f"saved={record.get(field)!r}; current={expected_value!r}"
            )

    request_size = record.get("comments_per_request")
    if not isinstance(request_size, int) or not (
        1 <= request_size <= COMMENTS_PER_REQUEST
    ):
        add_resume_error(
            f"Result {comment_id} has invalid comments_per_request "
            f"value {request_size!r}."
        )

    try:
        CommentExtraction.model_validate(record.get("extraction"))
    except Exception as error:
        add_resume_error(
            f"Result {comment_id} fails the current schema: {error}"
        )

    batch_id = record.get("batch_id")
    if not isinstance(batch_id, str) or not batch_id:
        add_resume_error(
            f"Result {comment_id} has no valid batch_id."
        )
    else:
        result_ids_by_batch.setdefault(batch_id, set()).add(comment_id)
        result_rows_by_batch.setdefault(batch_id, set()).add(
            int(expected_row["source_row"])
        )

    completed_ids.add(comment_id)

usage_by_batch = {}
for line_number, record in enumerate(existing_usage, start=1):
    if not isinstance(record, dict):
        add_resume_error(
            f"Usage line {line_number} is not a JSON object."
        )
        continue

    batch_id = record.get("batch_id")
    if not isinstance(batch_id, str) or not batch_id:
        add_resume_error(
            f"Usage line {line_number} has no valid batch_id."
        )
        continue
    if batch_id in usage_by_batch:
        add_resume_error(
            f"Duplicate usage record for batch {batch_id!r}."
        )
        continue
    usage_by_batch[batch_id] = record

    if record.get("run_signature_sha256") != run_signature_sha256:
        add_resume_error(
            f"Usage batch {batch_id} has a different or missing run signature."
        )
    if record.get("model") != MODEL:
        add_resume_error(f"Usage batch {batch_id} model mismatch.")
    if record.get("part_number") != PART_NUMBER:
        add_resume_error(f"Usage batch {batch_id} part-number mismatch.")
    if record.get("total_parts") != TOTAL_PARTS:
        add_resume_error(f"Usage batch {batch_id} total-parts mismatch.")
    if record.get("reasoning_effort") != REASONING_EFFORT:
        add_resume_error(
            f"Usage batch {batch_id} reasoning-effort mismatch."
        )

    raw_usage_comment_ids = record.get("comment_ids")
    if not isinstance(raw_usage_comment_ids, list):
        add_resume_error(
            f"Usage batch {batch_id} comment_ids must be a list."
        )
        usage_comment_ids = []
    else:
        usage_comment_ids = [
            str(comment_id) for comment_id in raw_usage_comment_ids
        ]

    raw_usage_source_rows = record.get("source_rows")
    if not isinstance(raw_usage_source_rows, list):
        add_resume_error(
            f"Usage batch {batch_id} source_rows must be a list."
        )
        usage_source_rows = set()
    else:
        try:
            usage_source_rows = {
                int(source_row) for source_row in raw_usage_source_rows
            }
        except (TypeError, ValueError):
            add_resume_error(
                f"Usage batch {batch_id} contains an invalid source row."
            )
            usage_source_rows = set()
    expected_batch_ids = result_ids_by_batch.get(batch_id)
    expected_batch_rows = result_rows_by_batch.get(batch_id)
    if expected_batch_ids is None:
        add_resume_error(
            f"Usage batch {batch_id} has no matching saved results."
        )
    else:
        if set(usage_comment_ids) != expected_batch_ids:
            add_resume_error(
                f"Usage batch {batch_id} comment IDs do not match its results."
            )
        if usage_source_rows != expected_batch_rows:
            add_resume_error(
                f"Usage batch {batch_id} source rows do not match its results."
            )

    if record.get("comment_count") != len(usage_comment_ids):
        add_resume_error(
            f"Usage batch {batch_id} comment_count is inconsistent."
        )
    usage_values = record.get("usage")
    if not isinstance(usage_values, dict):
        add_resume_error(
            f"Usage batch {batch_id} usage must be an object."
        )
        estimated_cost = None
    else:
        estimated_cost = usage_values.get("estimated_cost_usd")
    if not isinstance(estimated_cost, (int, float)) or estimated_cost < 0:
        add_resume_error(
            f"Usage batch {batch_id} has no valid estimated cost."
        )

for batch_id in sorted(set(result_ids_by_batch) - set(usage_by_batch)):
    add_resume_error(
        f"Saved result batch {batch_id} has no matching usage record."
    )

if resume_manifest is not None and isinstance(resume_manifest, dict):
    saved_status = resume_manifest.get("status")
    if saved_status in {"completed", "completed_with_audit_warnings"} and (
        len(completed_ids) != len(df_final)
    ):
        add_resume_error(
            f"Manifest status is {saved_status!r}, but only "
            f"{len(completed_ids):,} of {len(df_final):,} comments are saved."
        )

if resume_error_state["count"]:
    details = "\n".join(
        f"- {message}"
        for message in resume_error_state["examples"]
    )
    raise RuntimeError(
        "RESUME VALIDATION FAILED. No saved comment IDs were skipped, "
        "and the paid run is blocked.\n"
        f"Problems found: {resume_error_state['count']:,}\n"
        f"{details}"
    )

remaining_df = df_final[
    ~df_final["id"].astype(str).isin(completed_ids)
].copy()
remaining_rows = remaining_df.to_dict("records")
batches = make_batches(remaining_rows, COMMENTS_PER_REQUEST)

existing_cost = sum(
    record["usage"]["estimated_cost_usd"]
    for record in usage_by_batch.values()
)
global_usage_records, global_existing_cost = load_global_usage(
    OUTPUT_ROOT,
    expected_total_parts=TOTAL_PARTS,
)
resume_validation_status = (
    "passed" if resume_manifest is not None else "fresh_start"
)

run_plan = {
    **run_signature,
    "run_signature_sha256": run_signature_sha256,
    "resume_validation": resume_validation_status,
    "already_completed": len(completed_ids),
    "comments_remaining": len(remaining_rows),
    "requests_remaining": len(batches),
    "existing_estimated_cost_usd": round(existing_cost, 4),
    "global_existing_estimated_cost_usd": round(
        global_existing_cost, 4
    ),
    "global_max_cost_usd": MAX_TOTAL_COST_USD,
}

print("Resume validation:", resume_validation_status)
print(json.dumps(run_plan, indent=2))

confirmation = input(
    f"Type {EXPECTED_CONFIRMATION} to start or resume paid part "
    f"{PART_NUMBER}/{TOTAL_PARTS}: "
).strip()


Resume validation: fresh_start
{
  "method_version": "airbnb_extraction_v1.0",
  "partition_version": "airbnb_final_50000_five_equal_parts_v1",
  "part_number": 4,
  "total_parts": 5,
  "rows_per_part": 10000,
  "model": "gpt-5-mini-2025-08-07",
  "reasoning_effort": "medium",
  "comments_per_request": 10,
  "max_workers": 10,
  "max_attempts": 3,
  "request_timeout_seconds": 600.0,
  "requests_per_wave": 10,
  "input_rows": 10000,
  "config_sha256": "364430c438c0562829adf0a8f49577c49c2896791742655b25f318c4a8d7dbcb",
  "workflow_sha256": "668342a8db60240f906456b4ffced9aaf6b26fd908cc0a2f6bc988c43901eecb",
  "master_input_sha256": "6f95b3d3bccb6d31f8b57c4e9b8615f2193847bbb19cabb1f0695ec52fc3c5f9",
  "input_sha256": "d593fa86eb4290ef24a8bd56bfadabc24ef3d909f0542d5e65478d6bd1c35fc5",
  "prompt_sha256": "0d033b69fcff7c9aec4323635f7cce5de16aaf6ec619871cabeef3041f7cb3b5",
  "schema_sha256": "5a5796f3c3d5406429b6e32abe3f2620e9560fcbe4b713e0e6cb73027951d23d",
  "run_signature_sha256": "ed561926

## PAID PRODUCTION RUN — SELECTED PART ONLY
### Run this cell only after the plan above is correct.

In [6]:
if confirmation != EXPECTED_CONFIRMATION:
    print("Production run cancelled.")

elif not batches:
    print(f"All comments in part {PART_NUMBER} are already complete.")

else:
    run_started = time.perf_counter()
    run_started_utc = datetime.now(timezone.utc).isoformat()

    successful_requests = 0
    failed_requests = 0
    successful_comments = 0
    run_cost = 0.0
    stopped_for_budget = False
    fatal_error_record = None

    manifest = {
        **run_plan,
        "run_started_utc": run_started_utc,
        "status": "running",
    }
    MANIFEST_FILE.write_text(
        json.dumps(manifest, indent=2),
        encoding="utf-8",
    )

    for wave_start in range(0, len(batches), REQUESTS_PER_WAVE):
        wave = batches[wave_start : wave_start + REQUESTS_PER_WAVE]
        wave_number = wave_start // REQUESTS_PER_WAVE + 1
        total_waves = (len(batches) + REQUESTS_PER_WAVE - 1) // REQUESTS_PER_WAVE

        # Check the shared five-part budget before any request in this wave.
        global_usage_before_wave, global_cost_before_wave = load_global_usage(
            OUTPUT_ROOT,
            expected_total_parts=TOTAL_PARTS,
        )
        budget_reserve = estimate_wave_cost_reserve(
            len(wave),
            global_usage_before_wave,
            minimum_per_request_usd=(
                MIN_BUDGET_RESERVE_PER_REQUEST_USD
            ),
            observed_cost_multiplier=(
                OBSERVED_COST_RESERVE_MULTIPLIER
            ),
        )
        projected_global_cost = (
            global_cost_before_wave
            + budget_reserve["wave_reserve_usd"]
        )
        if projected_global_cost > MAX_TOTAL_COST_USD:
            stopped_for_budget = True
            print(
                f"\nStopped before wave {wave_number}: global cost is "
                f"${global_cost_before_wave:.2f} and the next-wave "
                f"reserve is ${budget_reserve['wave_reserve_usd']:.2f}, "
                f"which could exceed the ${MAX_TOTAL_COST_USD:.2f} limit."
            )
            break

        print(
            f"\nStarting wave {wave_number}/{total_waves} "
            f"({len(wave)} API requests; global cost "
            f"${global_cost_before_wave:.2f})..."
        )

        with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
            futures = {
                executor.submit(extract_one_batch, batch): batch["batch_id"]
                for batch in wave
            }

            with RESULTS_FILE.open("a", encoding="utf-8") as results_file, \
                 ERRORS_FILE.open("a", encoding="utf-8") as errors_file, \
                 USAGE_FILE.open("a", encoding="utf-8") as usage_file:

                for finished_number, future in enumerate(
                    as_completed(futures),
                    start=1,
                ):
                    result = future.result()

                    if result["status"] == "success":
                        for record in result["comment_records"]:
                            results_file.write(
                                json.dumps(record, ensure_ascii=False) + "\n"
                            )
                        results_file.flush()

                        usage_file.write(
                            json.dumps(
                                result["usage_record"],
                                ensure_ascii=False,
                            )
                            + "\n"
                        )
                        usage_file.flush()

                        successful_requests += 1
                        successful_comments += len(result["comment_records"])
                        run_cost += result["usage_record"]["usage"][
                            "estimated_cost_usd"
                        ]
                    else:
                        error_record = dict(result)
                        error_record.pop("status")
                        errors_file.write(
                            json.dumps(error_record, ensure_ascii=False) + "\n"
                        )
                        errors_file.flush()
                        failed_requests += 1

                        if result["status"] == "fatal_error":
                            fatal_error_record = error_record
                            for pending_future in futures:
                                pending_future.cancel()
                            print(
                                f"Fatal non-retryable error in "
                                f"{result['batch_id']}: "
                                f"{result['error_type']}. "
                                "Cancelling pending requests."
                            )
                            break

                    if finished_number % 10 == 0 or finished_number == len(wave):
                        total_done = len(completed_ids) + successful_comments
                        print(
                            f"Wave {wave_number}: {finished_number}/{len(wave)} requests | "
                            f"new comments={successful_comments:,} | "
                            f"total completed≈{total_done:,} | "
                            f"batch errors={failed_requests} | "
                            f"new cost=${run_cost:.4f}"
                        )

        if fatal_error_record is not None:
            print("Production run stopped for a non-retryable error.")
            break

        _, global_cost_after_wave = load_global_usage(
            OUTPUT_ROOT,
            expected_total_parts=TOTAL_PARTS,
        )
        print(
            f"Global five-part estimated cost after wave: "
            f"${global_cost_after_wave:.4f}"
        )
        if global_cost_after_wave >= MAX_TOTAL_COST_USD:
            stopped_for_budget = True
            print(
                f"The shared ${MAX_TOTAL_COST_USD:.2f} safety limit "
                "has been reached; no further wave will be submitted."
            )
            break

    run_seconds = time.perf_counter() - run_started
    run_finished_utc = datetime.now(timezone.utc).isoformat()
    global_usage_after_run, global_cost_after_run = load_global_usage(
        OUTPUT_ROOT,
        expected_total_parts=TOTAL_PARTS,
    )

    # Re-read the checkpoint file instead of assuming submitted requests
    # produced a complete run. Completion is based on saved, valid records.
    saved_after_run = read_jsonl(RESULTS_FILE)
    saved_after_run_by_id = {
        str(record["comment_id"]): record
        for record in saved_after_run
        if record.get("comment_id") is not None
    }
    expected_after_run_ids = set(df_final["id"].astype(str))
    returned_after_run_ids = set(saved_after_run_by_id)
    missing_after_run_ids = sorted(
        expected_after_run_ids - returned_after_run_ids
    )
    unexpected_after_run_ids = sorted(
        returned_after_run_ids - expected_after_run_ids
    )

    schema_errors_after_run = []
    for comment_id in sorted(
        expected_after_run_ids & returned_after_run_ids
    ):
        try:
            CommentExtraction.model_validate(
                saved_after_run_by_id[comment_id].get("extraction")
            )
        except Exception as error:
            schema_errors_after_run.append(
                {"comment_id": comment_id, "error": str(error)}
            )

    coverage_is_complete = (
        not missing_after_run_ids
        and not unexpected_after_run_ids
        and len(returned_after_run_ids) == len(expected_after_run_ids)
    )
    records_are_valid = not schema_errors_after_run

    if coverage_is_complete and records_are_valid and failed_requests == 0:
        run_status = "pending_audit"
    elif fatal_error_record is not None:
        run_status = "stopped_for_fatal_error"
    elif stopped_for_budget:
        run_status = "stopped_for_budget"
    else:
        run_status = "incomplete"

    manifest.update(
        {
            "run_finished_utc": run_finished_utc,
            "status": run_status,
            "stopped_for_budget": stopped_for_budget,
            "fatal_error": fatal_error_record,
            "new_successful_requests": successful_requests,
            "new_failed_requests": failed_requests,
            "new_successful_comments": successful_comments,
            "saved_unique_comment_count": len(returned_after_run_ids),
            "missing_comment_count": len(missing_after_run_ids),
            "unexpected_comment_count": len(unexpected_after_run_ids),
            "schema_error_count": len(schema_errors_after_run),
            "new_estimated_cost_usd": round(run_cost, 4),
            "part_total_estimated_cost_usd": round(
                existing_cost + run_cost, 4
            ),
            "global_total_estimated_cost_usd": round(
                global_cost_after_run, 4
            ),
            "global_successful_usage_records": len(
                global_usage_after_run
            ),
            "wall_clock_hours": round(run_seconds / 3600, 3),
        }
    )
    MANIFEST_FILE.write_text(
        json.dumps(manifest, indent=2),
        encoding="utf-8",
    )

    print("\n--------------- PRODUCTION RUN SUMMARY ---------------")
    print("Run status:", run_status)
    print("New successful comments:", successful_comments)
    print("Saved unique comments:", len(returned_after_run_ids))
    print("Comments still missing:", len(missing_after_run_ids))
    print("Unexpected comment IDs:", len(unexpected_after_run_ids))
    print("Schema-invalid saved records:", len(schema_errors_after_run))
    print("New failed batch requests:", failed_requests)
    print("Wall-clock hours:", round(run_seconds / 3600, 2))
    print("New estimated cost: $", round(run_cost, 4))
    print(
        "Part total estimated cost: $",
        round(existing_cost + run_cost, 4),
    )
    print(
        "Global five-part estimated cost: $",
        round(global_cost_after_run, 4),
    )
    print("Manifest saved:", MANIFEST_FILE.name)
    if run_status == "pending_audit":
        print("Run the final audit cell to determine completed status.")
    else:
        print("The extraction is not complete. Resume before final analysis.")



Starting wave 1/100 (10 API requests; global cost $42.28)...
Wave 1: 10/10 requests | new comments=100 | total completed≈100 | batch errors=0 | new cost=$0.1495
Global five-part estimated cost after wave: $42.4322

Starting wave 2/100 (10 API requests; global cost $42.43)...
Wave 2: 10/10 requests | new comments=200 | total completed≈200 | batch errors=0 | new cost=$0.2842
Global five-part estimated cost after wave: $42.5670

Starting wave 3/100 (10 API requests; global cost $42.57)...
Wave 3: 10/10 requests | new comments=300 | total completed≈300 | batch errors=0 | new cost=$0.4276
Global five-part estimated cost after wave: $42.7104

Starting wave 4/100 (10 API requests; global cost $42.71)...
Wave 4: 10/10 requests | new comments=400 | total completed≈400 | batch errors=0 | new cost=$0.5646
Global five-part estimated cost after wave: $42.8474

Starting wave 5/100 (10 API requests; global cost $42.85)...
Wave 5: 10/10 requests | new comments=500 | total completed≈500 | batch errors

## Audit the selected part and merge only when all five parts are ready
### This cell can be rerun safely after the production run.

In [7]:
raw_results = read_jsonl(RESULTS_FILE)

# Keep only the latest record for each comment ID.
result_by_id = {
    str(record["comment_id"]): record
    for record in raw_results
}
duplicate_saved_line_count = len(raw_results) - len(result_by_id)
all_results = sorted(
    result_by_id.values(),
    key=lambda record: int(record["source_row"]),
)

expected_ids = set(df_final["id"].astype(str))
returned_ids = set(result_by_id)
missing_ids = sorted(expected_ids - returned_ids)
unexpected_ids = sorted(returned_ids - expected_ids)

schema_errors = []
evidence_errors = []
finding_rows = []
review_rows = []

for record in all_results:
    comment_id = str(record["comment_id"])
    try:
        validated = CommentExtraction.model_validate(record["extraction"])
    except Exception as error:
        schema_errors.append(
            {
                "comment_id": comment_id,
                "error": str(error),
            }
        )
        continue

    review_rows.append(
        {
            "source_row": record["source_row"],
            "part_number": PART_NUMBER,
            "listing_id": record["listing_id"],
            "comment_id": comment_id,
            "date": record["date"],
            "year": record["year"],
            "listing_comment_count": record["listing_comment_count"],
            "listing_activity": record["listing_activity"],
            "comments_original": record["comments_original"],
            "comments_clean": record["comments_clean"],
            "finding_count": len(validated.findings),
            "batch_id": record["batch_id"],
            "attempt": record["attempt"],
        }
    )

    for finding_number, finding in enumerate(validated.findings, start=1):
        quote_is_exact = finding.evidence_quote in record["comments_clean"]
        if not quote_is_exact:
            evidence_errors.append(
                {
                    "comment_id": comment_id,
                    "finding_number": finding_number,
                    "evidence_quote": finding.evidence_quote,
                }
            )

        finding_rows.append(
            {
                "source_row": record["source_row"],
                "part_number": PART_NUMBER,
                "listing_id": record["listing_id"],
                "comment_id": comment_id,
                "date": record["date"],
                "year": record["year"],
                "listing_comment_count": record["listing_comment_count"],
                "listing_activity": record["listing_activity"],
                "finding_number": finding_number,
                "aspect": finding.aspect,
                "object": finding.object,
                "observation": finding.observation,
                "aspect_score": finding.aspect_score,
                "severity_score": finding.severity_score,
                "evidence_quote": finding.evidence_quote,
                "evidence_quote_exact": quote_is_exact,
                "finding_category": (
                    "Strength"
                    if finding.aspect_score > 0
                    else "Problem"
                    if finding.aspect_score < 0
                    else "Neutral"
                ),
                "manual_review": finding.aspect == "Other",
                "comments_clean": record["comments_clean"],
            }
        )

findings_df = pd.DataFrame(finding_rows)
review_summary_df = pd.DataFrame(review_rows)

findings_df.to_csv(FINDINGS_FILE, index=False)
review_summary_df.to_csv(REVIEW_SUMMARY_FILE, index=False)

usage_records = read_jsonl(USAGE_FILE)
usage_by_batch = {record["batch_id"]: record for record in usage_records}
usage_records = list(usage_by_batch.values())

total_cost = sum(
    record.get("usage", {}).get("estimated_cost_usd", 0.0)
    for record in usage_records
)
global_usage_records, global_total_cost = load_global_usage(
    OUTPUT_ROOT,
    expected_total_parts=TOTAL_PARTS,
)

coverage_and_schema_complete = (
    len(all_results) == len(df_final)
    and not missing_ids
    and not unexpected_ids
    and duplicate_saved_line_count == 0
    and not schema_errors
)

if coverage_and_schema_complete and not evidence_errors:
    completion_status = "completed"
elif coverage_and_schema_complete:
    completion_status = "completed_with_audit_warnings"
else:
    completion_status = "incomplete"

audit = {
    "method_version": METHOD_VERSION,
    "partition_version": PARTITION_VERSION,
    "part_number": PART_NUMBER,
    "total_parts": TOTAL_PARTS,
    "completion_status": completion_status,
    "is_complete": completion_status == "completed",
    "expected_comments": len(df_final),
    "returned_comments": len(all_results),
    "missing_comment_count": len(missing_ids),
    "unexpected_comment_count": len(unexpected_ids),
    "duplicate_saved_line_count": duplicate_saved_line_count,
    "schema_error_count": len(schema_errors),
    "finding_count": len(findings_df),
    "evidence_quote_error_count": len(evidence_errors),
    "evidence_quote_pass_rate": (
        1.0
        if len(findings_df) == 0
        else round(1 - len(evidence_errors) / len(findings_df), 6)
    ),
    "other_finding_count": (
        0
        if findings_df.empty
        else int((findings_df["aspect"] == "Other").sum())
    ),
    "successful_usage_records": len(usage_records),
    "part_estimated_total_cost_usd": round(total_cost, 4),
    "global_estimated_total_cost_usd": round(global_total_cost, 4),
    "global_successful_usage_records": len(global_usage_records),
    "global_max_cost_usd": MAX_TOTAL_COST_USD,
    "missing_comment_ids": missing_ids[:100],
    "unexpected_comment_ids": unexpected_ids[:100],
    "schema_errors": schema_errors[:100],
    "evidence_errors": evidence_errors[:100],
}

AUDIT_FILE.write_text(
    json.dumps(audit, indent=2, ensure_ascii=False),
    encoding="utf-8",
)

# The audit is the only step allowed to assign the final completed status.
if MANIFEST_FILE.exists():
    try:
        manifest = json.loads(MANIFEST_FILE.read_text(encoding="utf-8"))
    except json.JSONDecodeError as error:
        raise ValueError(
            f"Cannot update invalid manifest {MANIFEST_FILE.name}: {error}"
        ) from error

    manifest.update(
        {
            "status": completion_status,
            "audit_finished_utc": datetime.now(timezone.utc).isoformat(),
            "audit_missing_comment_count": len(missing_ids),
            "audit_unexpected_comment_count": len(unexpected_ids),
            "audit_schema_error_count": len(schema_errors),
            "audit_evidence_quote_error_count": len(evidence_errors),
        }
    )
    MANIFEST_FILE.write_text(
        json.dumps(manifest, indent=2),
        encoding="utf-8",
    )

print(json.dumps(audit, indent=2, ensure_ascii=False))
print(f"\nPART {PART_NUMBER} COMPLETION STATUS:", completion_status)
if completion_status == "completed":
    print(f"All {ROWS_PER_PART:,} comments in part {PART_NUMBER} passed audit.")
elif completion_status == "completed_with_audit_warnings":
    print("Coverage is complete, but audit warnings require review.")
else:
    print("The extraction is incomplete. Resume it before final analysis.")
print("\nAnalysis-ready findings:", FINDINGS_FILE.name)
print("Review-level summary:", REVIEW_SUMMARY_FILE.name)
print("Audit report:", AUDIT_FILE.name)

merge_audit = merge_final_parts(
    df_master,
    OUTPUT_ROOT,
    MERGED_OUTPUT_DIR,
    total_parts=TOTAL_PARTS,
    rows_per_part=ROWS_PER_PART,
    comment_model=CommentExtraction,
    method_version=METHOD_VERSION,
    max_total_cost_usd=MAX_TOTAL_COST_USD,
)
print("\n--------------- FIVE-PART MERGE STATUS ---------------")
print(json.dumps(merge_audit, indent=2, ensure_ascii=False))
if merge_audit["completion_status"] == "waiting_for_parts":
    print(
        "Complete and audit the remaining parts:",
        merge_audit["parts_remaining"],
    )
elif merge_audit.get("coverage_complete"):
    print("Merged 50,000-comment outputs:", MERGED_OUTPUT_DIR)


{
  "method_version": "airbnb_extraction_v1.0",
  "partition_version": "airbnb_final_50000_five_equal_parts_v1",
  "part_number": 4,
  "total_parts": 5,
  "completion_status": "completed_with_audit_warnings",
  "is_complete": false,
  "expected_comments": 10000,
  "returned_comments": 10000,
  "missing_comment_count": 0,
  "unexpected_comment_count": 0,
  "duplicate_saved_line_count": 0,
  "schema_error_count": 0,
  "finding_count": 40327,
  "evidence_quote_error_count": 954,
  "evidence_quote_pass_rate": 0.976343,
  "other_finding_count": 1965,
  "successful_usage_records": 1000,
  "part_estimated_total_cost_usd": 14.1895,
  "global_estimated_total_cost_usd": 56.4723,
  "global_successful_usage_records": 4000,
  "global_max_cost_usd": 80.0,
  "missing_comment_ids": [],
  "unexpected_comment_ids": [],
  "schema_errors": [],
  "evidence_errors": [
    {
      "comment_id": "876899844346115916",
      "finding_number": 3,
      "evidence_quote": "the views are so awesome from the two(!) 